> 在线讲义：[第 05 章 · 用几个数字估算一次模型执行：算和读各要多久，实测为什么达不到](https://wikieden.github.io/ai-infra-from-constraints/lessons/05-estimate.html) · 本书《从约束推导 AI Infra》

# 第 5 章 · 用几个数字估算一次模型执行：算和读各要多久，实测为什么达不到

配套讲义：仓库里 `ai-infra/notes/lessons/05-estimate.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 存、读、算 | $b_W N$，$2BN$ | `step_bounds(1)` | 70 GB、140 GFLOPs |
| 第二步 两个下界 | $R/\beta$，$F/\Pi$ | `step_bounds(1)` | 20.90 ms、0.1415 ms，约 148 倍 |
| 第二步 三种改法 | 改 $\Pi$、$\beta$、$b_W$ | `step_bounds(1, pi=…, beta=…, bw=…)` | 20.90、10.45、10.45 ms |
| 第三步 $B = 8$ | 分摊与吞吐 | `batch_view(8)` | 1.13 ms，2.61 ms / 输出，383 token/s |
| 第三步 转折点 | $b_W\Pi/(2\beta)$ | `b_star()` | 147.7 |
| 第五步 Qwen3-8B 下界与 MBU | $(15.14 + 0.302)/1.792$ | `qwen_lower_ms()`、`mbu(26.5)` | 8.62 ms、33% |

数字取自 AII 第 1 章 §1.3：例题 1-1（70B、每参数 1 字节、H100 SXM 989.4 TFLOP/s 与 3.35 TB/s）、
Qwen3-8B 在 RTX PRO 6000（1.792 TB/s）上的读取下界与实测输出间隔。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
N = 70e9  # 参数数（手算近似）
PI = 989.4e12  # H100 SXM BF16 稠密矩阵峰值，FLOP/s
BETA = 3.35e12  # H100 SXM HBM 带宽，bytes/s

**函数 `step_bounds`**：一步生成的下界（ms）：(读权重, 矩阵计算, 两者取大)。

In [ ]:
def step_bounds(B: int, n: float = N, bw: float = 1, pi: float = PI, beta: float = BETA) -> tuple[float, float, float]:
    """一步生成的下界（ms）：(读权重, 矩阵计算, 两者取大)。"""
    read_ms = bw * n / beta * 1e3  # 读一遍权重，和 B 无关
    compute_ms = 2 * B * n / pi * 1e3  # 每个请求 2N 次运算
    return read_ms, compute_ms, max(read_ms, compute_ms)  # 充分重叠：取较大者

**函数 `batch_view`**：batch 为 B 时：(一步下界 ms, 每个输出分摊 ms, 吞吐上界 token/s)。

In [ ]:
def batch_view(B: int) -> tuple[float, float, float]:
    """batch 为 B 时：(一步下界 ms, 每个输出分摊 ms, 吞吐上界 token/s)。"""
    lower = step_bounds(B)[2]
    return lower, lower / B, B / lower * 1e3

**函数 `b_star`**：计算追上读取的 batch：B* = b_W × Π / (2β)。

In [ ]:
def b_star(bw: float = 1, pi: float = PI, beta: float = BETA) -> float:
    """计算追上读取的 batch：B* = b_W × Π / (2β)。"""
    return bw * pi / (2 * beta)

**函数 `qwen_lower_ms`**：Qwen3-8B 一步 decode 读主要权重 + 旧 KV 的下界（ms）。

In [ ]:
def qwen_lower_ms(weights_gb: float = 15.14, kv_gb: float = 0.302, beta_tb: float = 1.792) -> float:
    """Qwen3-8B 一步 decode 读主要权重 + 旧 KV 的下界（ms）。"""
    return (weights_gb + kv_gb) / (beta_tb * 1e3) * 1e3

**函数 `mbu`**：带宽利用率 = 读取下界 ÷ 实测时间。

In [ ]:
def mbu(measured_ms: float, lower_ms: float | None = None) -> float:
    """带宽利用率 = 读取下界 ÷ 实测时间。"""
    return (lower_ms if lower_ms is not None else qwen_lower_ms()) / measured_ms

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步：70B、每参数 1 字节，batch = 1")
print(f"  存 = 读 = {N / 1e9:.0f} GB；算 = 2N = {2 * N / 1e9:.0f} GFLOPs")

In [ ]:
print("\n第二步：两个下界（H100 SXM）")
r, c, low = step_bounds(1)
print(f"  读权重 {r:.2f} ms，矩阵计算 {c:.4f} ms，读是算的 {r / c:.0f} 倍 → 下界 {low:.2f} ms > 10 ms")
print(f"  算力翻倍：下界 {step_bounds(1, pi=2 * PI)[2]:.2f} ms（计算 {step_bounds(1, pi=2 * PI)[1]:.4f} ms）")
print(f"  带宽翻倍：下界 {step_bounds(1, beta=2 * BETA)[2]:.2f} ms")
print(f"  权重减半：下界 {step_bounds(1, bw=0.5)[2]:.2f} ms")

In [ ]:
print("\n第三步：批内复用")
for B in (1, 8):
    lower, per, tput = batch_view(B)
    print(f"  B = {B}：计算 {step_bounds(B)[1]:.2f} ms，一步 {lower:.2f} ms，每输出分摊 {per:.2f} ms，吞吐 {tput:.1f} token/s")
print(f"  转折点 B* = b_W × Π / (2β) = {b_star():.1f}；Π/β = {PI / BETA:.0f} FLOP/字节")

In [ ]:
print("\n第五步：Qwen3-8B 在 RTX PRO 6000 上")
low = qwen_lower_ms()
print(f"  下界 (15.14 + 0.302) GB ÷ 1.792 TB/s = {low:.2f} ms")
for t in (26.5, 12.1, 10.7):
    print(f"  实测 {t} ms → MBU {mbu(t):.0%}")

## 3. 代码里才看得到的

1. 吞吐用 `B / lower_ms * 1e3`，单位是 token/s；把毫秒忘了乘 1000，会差三个数量级。
2. 原文的 47.9 和 383 token/s 是由 20.90 ms 算的；用不四舍五入的 20.896 ms 算出来是 47.86 和 382.9，测试里放宽到一位小数。
3. `b_star()` 里 $N$ 被约掉了：同一张卡上，7B 和 70B 的转折点一样，都只看 $b_W$ 和 $\Pi/\beta$。

## 4. 自己改一改

- 把 `step_bounds` 的 `bw` 改成 2（BF16），看读取下界和 `b_star()` 怎么变。
- 用 `batch_view(B)` 扫 $B$ = 1、64、128、256、512，画出吞吐曲线，和图 5-3 比。
- 把 `mbu` 的实测时间换成 12.1 和 10.7，看利用率怎么一步步上升。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 存读算，分三算；读比算慢百来番；
> 加算力，白费钱；加带宽、少读数，才见效；
> 一车拉八人，油钱分摊路不短；
> 先算极限再看差，漏项开销分开查。